# Knesset Committee Session Segmenter
**RAG-augmented few-shot segmentation using a parliamentary-tuned Gemma 4B model.**

Requirements: Runtime → Change runtime type → **T4 GPU**

In [9]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q sentence-transformers transformers accelerate bitsandbytes>=0.46.1

In [10]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ⬇ Set this to the folder in your Drive that contains:
#   committee_data/  (the JSON session files)
#   committee_subject_labeling_signal.csv
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/anlp'  # ← change if needed

import os, sys
os.chdir(DRIVE_PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/anlp
Files: ['part_1', 'part_0', 'committee_subject_labeling_signal.csv', '=0.46.1']
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/anlp
Files: ['part_1', 'part_0', 'committee_subject_labeling_signal.csv', '=0.46.1', 'results']


In [ ]:
# ── Cell 3: Config ────────────────────────────────────────────────────────────

# RAG / chunking settings
CHUNK_SIZE     = 40   # utterances per LLM call
CHUNK_OVERLAP  = 5
TOP_K_EXAMPLES = 1   # gold examples in prompt

GOLD_CSV    = 'committee_subject_labeling_signal.csv'
DATA_GLOB   = '**/*.json'
EMBED_MODEL = 'intfloat/multilingual-e5-small'
LLM_MODEL   = 'abanwild/nation-parliamentary-gemma4-e4b-grpo-lora'
OUT_DIR     = 'results'

# Evaluation settings
BOUNDARY_TOLERANCE = 3   # ±N utterances to count a boundary as correct
MAX_EVAL_SESSIONS  = None  # set to e.g. 10 for a quick smoke-test; None = all gold sessions

In [ ]:
# ── Cell 4: Load gold data + build retrieval index (all sessions) ─────────────
import os, csv, json, glob as _glob
import numpy as np
from sentence_transformers import SentenceTransformer

def load_gold(path):
    gold = {}
    with open(path, encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            gold.setdefault(row['doc_id'], []).append({
                'from': int(row['utterance_from']),
                'to':   int(row['utterance_to']),
                'label': row['subject_label'],
                'tags':  row['tags'],
                'session_title': row['session_title'],
            })
    return gold

def build_path_map(pattern):
    return {
        os.path.splitext(os.path.basename(p))[0]: p
        for p in _glob.glob(pattern, recursive=True)
    }

def load_session(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

gold     = load_gold(GOLD_CSV)
path_map = build_path_map(DATA_GLOB)
print(f'Gold sessions: {len(gold)} | Corpus sessions: {len(path_map)}')

# Embedder on CPU — keeps full GPU free for the LLM
embedder = SentenceTransformer(EMBED_MODEL, device='cpu')

def embed(texts, prefix='passage'):
    return embedder.encode([f'{prefix}: {t}' for t in texts],
                           normalize_embeddings=True, show_progress_bar=False)

# Build index over ALL gold sessions (leave-one-out happens at query time)
gold_ids, gold_docs = [], {}
gold_texts = []
for doc_id, segs in gold.items():
    path = path_map.get(doc_id)
    if not path:
        continue
    doc = load_session(path)
    gold_docs[doc_id] = doc
    title = doc.get('title', '')
    tags  = ' | '.join(s['tags'] for s in segs)
    gold_ids.append(doc_id)
    gold_texts.append(f'{title} {tags}')

print(f'Building index over {len(gold_ids)} sessions …')
gold_vecs = embed(gold_texts)
print('Index ready.')

In [ ]:
# ── Cell 5: Load LLM (4-bit NF4, T4 GPU) ─────────────────────────────────────
import torch
import gc
from transformers import pipeline, BitsAndBytesConfig

# Free any stale GPU memory before loading the model
gc.collect()
torch.cuda.empty_cache()
print(f'Free VRAM before load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

pipe = pipeline(
    'text-generation',
    model=LLM_MODEL,
    model_kwargs={'quantization_config': quant_config,
                  'dtype': torch.bfloat16},
    device_map='auto',
)

print(f'Model loaded.')
print(f'Free VRAM after load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

In [ ]:
# ── Cell 6: Segmentation helpers ──────────────────────────────────────────────
import re, json

SYSTEM_PROMPT = """אתה מסייע בניתוח פרוטוקולי ועדת הכנסת.
תפקידך: לחלק קטע מפרוטוקול לנושאים נפרדים ולזהות את גבולות כל נושא.

כללים:
- כל קטע מתחיל במספר האמירה הראשונה שעוסקת בנושא חדש.
- כל קטע מסתיים במספר האמירה האחרונה לפני המעבר לנושא הבא.
- ציין שם נושא קצר בעברית (עד 15 מילה).
- ציין תגיות מפתח בעברית המופרדות ב-|  (ישויות, חוקים, גופים, מושגים מרכזיים).
- החזר JSON בלבד — מערך של אובייקטים עם המפתחות: from, to, subject_label, tags."""

def retrieve_examples(query_text, exclude_id=None, k=TOP_K_EXAMPLES):
    q    = embed([query_text], prefix='query')[0]
    sims = gold_vecs @ q
    # leave-one-out: mask out the session being segmented
    for i, gid in enumerate(gold_ids):
        if gid == exclude_id:
            sims[i] = -1.0
    top = np.argsort(sims)[::-1][:k]
    return [(gold_docs[gold_ids[i]], gold[gold_ids[i]], float(sims[i])) for i in top]

def format_example(doc, segs):
    lines = [f"=== דוגמה: {doc.get('title', doc['doc_id'])[:80]} ==="]
    for seg in segs:
        lines.append(f"\n--- אמירות {seg['from']}–{seg['to']} ---")
        lines.append(f"נושא:  {seg['label']}")
        lines.append(f"תגיות: {seg['tags']}")
        snippet = [u for u in doc['utterances']
                   if seg['from'] <= int(u['id']) <= min(seg['from']+3, seg['to'])]
        for u in snippet:
            lines.append(f"[{u['id']}] {u.get('speaker','')}: {u['text'][:120].replace(chr(10),' ')}")
        lines.append('  …')
    return '\n'.join(lines)

def build_prompt(chunk_utts, examples):
    parts = ['להלן דוגמאות לאופן החלוקה לנושאים (שים לב לתגיות — הן מגדירות את הנושא):\n']
    for doc, segs, sim in examples:
        parts.append(format_example(doc, segs))
    start, end = int(chunk_utts[0]['id']), int(chunk_utts[-1]['id'])
    parts.append(f'\n=== פרוטוקול לחלוקה (אמירות {start}–{end}) ===')
    for u in chunk_utts:
        text = u.get('text', '').replace('\n', ' ').strip()
        parts.append(f"[{u['id']}] {u.get('speaker','')}: {text}")
    parts.append(
        f'\nחלק את האמירות {start}–{end} לנושאים.'
        ' החזר JSON בלבד — מערך:'
        ' [{"from": N, "to": M, "subject_label": "...", "tags": "תגית1 | תגית2 | ..."}]'
        ' ללא שום טקסט נוסף.'
    )
    return '\n'.join(parts)

def call_model(prompt):
    messages = [{'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + prompt}]
    out = pipe(messages, max_new_tokens=1024, do_sample=False)
    generated = out[0]['generated_text']
    if isinstance(generated, list):
        return generated[-1]['content'].strip()
    return str(generated).strip()

def parse_response(text):
    text = re.sub(r'^```[a-z]*\n?', '', text.strip())
    text = re.sub(r'\n?```$', '', text.strip())
    try:
        result = json.loads(text)
        if isinstance(result, list): return result
    except json.JSONDecodeError:
        pass
    m = re.search(r'\[.*\]', text, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    print(f'  [WARN] Could not parse response:\n{text[:300]}')
    return []

def chunk_utterances(utterances, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks, i = [], 0
    while i < len(utterances):
        chunks.append(utterances[i: i + size])
        i += size - overlap
    return chunks

def merge_chunks(chunk_results, utterances):
    boundaries = set()
    for segs in chunk_results:
        for s in segs:
            try: boundaries.add(int(s['from']))
            except: pass
    if not boundaries: return []
    sorted_b = sorted(boundaries)
    max_id   = int(utterances[-1]['id'])
    label_map, tags_map = {}, {}
    for segs in chunk_results:
        for s in segs:
            try:
                f = int(s['from'])
                if f not in label_map:
                    label_map[f] = s.get('subject_label', '')
                    tags_map[f]  = s.get('tags', '')
            except: pass
    spans = []
    for idx, b in enumerate(sorted_b):
        end = sorted_b[idx+1] - 1 if idx+1 < len(sorted_b) else max_id
        spans.append({'from': b, 'to': end,
                      'subject_label': label_map.get(b, ''),
                      'tags':          tags_map.get(b, '')})
    return spans

def segment_one(doc_id):
    """Segment a single session. Returns list of predicted span dicts."""
    path = path_map.get(doc_id)
    if not path:
        print(f'  [SKIP] {doc_id} — JSON not found')
        return []
    doc        = load_session(path)
    utterances = doc['utterances']
    title      = doc.get('title', doc_id)
    query_text = title + ' ' + ' '.join(u['text'][:50] for u in utterances[:5])
    examples   = retrieve_examples(query_text, exclude_id=doc_id)
    chunks     = chunk_utterances(utterances)
    chunk_results = []
    for chunk in chunks:
        prompt = build_prompt(chunk, examples)
        raw    = call_model(prompt)
        chunk_results.append(parse_response(raw))
    return merge_chunks(chunk_results, utterances)

print('Helpers loaded.')

In [ ]:
# ── Cell 7: Evaluation loop (leave-one-out over all gold sessions) ─────────────
import time, os
import pandas as pd
from IPython.display import display, clear_output

os.makedirs(OUT_DIR, exist_ok=True)

eval_ids = list(gold.keys())
if MAX_EVAL_SESSIONS:
    eval_ids = eval_ids[:MAX_EVAL_SESSIONS]

print(f'Running evaluation on {len(eval_ids)} sessions '
      f'(tolerance=±{BOUNDARY_TOLERANCE} utterances) …\n')

all_rows   = []   # flat prediction rows for CSV
eval_stats = []   # per-session metrics

for idx, doc_id in enumerate(eval_ids):
    t0 = time.time()
    print(f'[{idx+1}/{len(eval_ids)}] {doc_id}', end=' … ', flush=True)

    pred_segs = segment_one(doc_id)
    gold_segs = gold[doc_id]

    # ── boundary eval ──────────────────────────────────────────────────────────
    # Skip the very first utterance id — every session starts there, not a real boundary
    first_id   = min(s['from'] for s in gold_segs)
    gold_bounds = {s['from'] for s in gold_segs if s['from'] != first_id}
    pred_bounds = {s['from'] for s in pred_segs if s['from'] != first_id}

    def matched(a_set, b_set, tol):
        return sum(1 for a in a_set if any(abs(a - b) <= tol for b in b_set))

    tp_p = matched(pred_bounds, gold_bounds, BOUNDARY_TOLERANCE)  # predicted that hit gold
    tp_g = matched(gold_bounds, pred_bounds, BOUNDARY_TOLERANCE)  # gold that were found

    precision = tp_p / len(pred_bounds) if pred_bounds else 0.0
    recall    = tp_g / len(gold_bounds) if gold_bounds else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if precision + recall > 0 else 0.0)

    false_splits = len(pred_bounds) - tp_p   # predicted but no nearby gold
    false_merges = len(gold_bounds) - tp_g   # gold but no nearby predicted

    elapsed = time.time() - t0
    print(f'gold={len(gold_bounds)} pred={len(pred_bounds)} '
          f'P={precision:.2f} R={recall:.2f} F1={f1:.2f} '
          f'splits={false_splits} merges={false_merges}  ({elapsed:.0f}s)')

    eval_stats.append({
        'doc_id':        doc_id,
        'gold_bounds':   len(gold_bounds),
        'pred_bounds':   len(pred_bounds),
        'precision':     precision,
        'recall':        recall,
        'f1':            f1,
        'false_splits':  false_splits,
        'false_merges':  false_merges,
    })

    # collect flat rows for output CSV
    for s in pred_segs:
        all_rows.append({
            'doc_id':                  doc_id,
            'utterance_from':          s['from'],
            'utterance_to':            s['to'],
            'subject_label_predicted': s['subject_label'],
            'tags_predicted':          s['tags'],
        })

# Save all predictions
pred_path = os.path.join(OUT_DIR, 'all_predictions.csv')
pd.DataFrame(all_rows).to_csv(pred_path, index=False, encoding='utf-8-sig')
print(f'\nPredictions saved → {pred_path}')

stats_df = pd.DataFrame(eval_stats)
stats_path = os.path.join(OUT_DIR, 'eval_stats.csv')
stats_df.to_csv(stats_path, index=False, encoding='utf-8-sig')
print(f'Per-session stats saved → {stats_path}')

In [ ]:
# ── Cell 8: Aggregate scores ──────────────────────────────────────────────────
import pandas as pd
from IPython.display import display

stats_df = pd.read_csv(os.path.join(OUT_DIR, 'eval_stats.csv'))

macro_p  = stats_df['precision'].mean()
macro_r  = stats_df['recall'].mean()
macro_f1 = stats_df['f1'].mean()
total_splits  = stats_df['false_splits'].sum()
total_merges  = stats_df['false_merges'].sum()
total_gold    = stats_df['gold_bounds'].sum()
total_pred    = stats_df['pred_bounds'].sum()

print('═' * 50)
print(f'  Sessions evaluated : {len(stats_df)}')
print(f'  Macro Precision    : {macro_p:.3f}')
print(f'  Macro Recall       : {macro_r:.3f}')
print(f'  Macro F1           : {macro_f1:.3f}')
print(f'  Total gold bounds  : {total_gold}')
print(f'  Total pred bounds  : {total_pred}')
print(f'  False splits       : {total_splits}  (predicted, no nearby gold)')
print(f'  False merges       : {total_merges}  (gold missed by model)')
print('═' * 50)

print('\nPer-session breakdown (sorted by F1):')
display(stats_df.sort_values('f1').to_string(index=False))